In [ ]:
# KALMAN OPEN REVALIDATION v3.0 — FROZEN ORIGINAL CONTRACT / ONE CELL
# Research only. Rebuilds all OPEN candidates from current canonical IEX features
# using the exact source-of-truth contract introduced in commit f56d51c795989c33961ee98cfd11fffefae235b5.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd, numpy as np, math, json, hashlib
from datetime import datetime, timezone

ROOT=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results/open_revalidation_v1")
AUD=ROOT/"open_revalidation_trade_audit.parquet"
OUT=ROOT/"frozen_original_contract_v3_0"
OUT.mkdir(parents=True,exist_ok=True)

a=pd.read_parquet(AUD).copy()
print("[AUDIT]",a.shape)

POLICIES={
"OPEN_NEG_0BP_0M":(0,lambda r:r.position_return_open<=0.0),
"OPEN_NEG_20BP_0M":(0,lambda r:r.position_return_open<=-0.002),
"OPEN_FLIP_0M":(0,lambda r:r.position_return_prev_close>0.0 and r.position_return_open<=0.0),
"OPEN_NEG_5M_CONFIRM":(5,lambda r:r.position_return_open<=0.0 and r.open_momentum_5m<=0.0),
"OPEN_FLIP_5M_CONFIRM":(5,lambda r:r.position_return_prev_close>0.0 and r.position_return_open<=0.0 and r.open_momentum_5m<=0.0),
"OPEN_GAP_5M_CONFIRM":(5,lambda r:r.overnight_gap_return<0.0 and r.position_return_5m<=0.0 and r.open_momentum_5m<=0.0),
"OPEN_GIVEBACK_5M":(5,lambda r:r.position_return_prev_close>=0.005 and r.giveback_prev_close_to_5m>=0.007 and r.open_momentum_5m<=0.0),
"OPEN_NEG_15M_CONFIRM":(15,lambda r:r.position_return_open<=0.0 and r.open_momentum_15m<=0.0),
}
EXITPX={0:"open_0_price_iex",5:"open_5_price_iex",15:"open_15_price_iex"}

need=["net_return","weight","entry_price_iex","fixed4_exit_price_iex","reconstructed_fixed4_raw_return",
"position_return_prev_close","position_return_open","position_return_5m","position_return_15m",
"overnight_gap_return","open_momentum_5m","open_momentum_15m","giveback_prev_close_to_5m"]
for c in need:
    if c not in a: raise RuntimeError("missing "+c)

ready=a["revalidation_data_ready"].fillna(False).astype(bool)
# Strictly require all source features and fixed4 raw reconstruction.
feature_complete=a[need].notna().all(axis=1)
eligible=ready & feature_complete
print("[READY FLAG]",int(ready.sum()),"/",len(a))
print("[ELIGIBLE ORIGINAL CONTRACT]",int(eligible.sum()),"/",len(a))
print("[UNRESOLVED]",len(a)-int(eligible.sum()))

# Rebuild candidate returns from authoritative historical FIXED_4 net + same-feed IEX delta.
for p,(lat,fn) in POLICIES.items():
    tc=p+"__v3_trigger"; nc=p+"__v3_net_return"; xc=p+"__v3_exit_price"
    a[tc]=False
    a[nc]=pd.to_numeric(a["net_return"],errors="coerce")
    a[xc]=np.nan
    for idx in a.index[eligible]:
        r=a.loc[idx]
        trig=bool(fn(r))
        a.at[idx,tc]=trig
        if trig:
            px=float(r[EXITPX[lat]])
            cand_raw=px/float(r.entry_price_iex)-1.0
            fixed_raw=float(r.reconstructed_fixed4_raw_return)
            a.at[idx,nc]=float(r.net_return)+float(r.weight)*(cand_raw-fixed_raw)
            a.at[idx,xc]=px

# Gold-20 regression against original pre-v2.4 outputs.
backs=sorted(ROOT.glob("open_revalidation_trade_audit.pre_derived_v2_4_*.parquet"))
if not backs: raise RuntimeError("pre-v2.4 backup not found")
pre=pd.read_parquet(backs[-1])
goldmask=pre["reconstructed_fixed4_net_return"].notna()
if int(goldmask.sum())!=20: raise RuntimeError(f"gold20 recovery failed: {int(goldmask.sum())}")
keys=[c for c in ["fold","symbol","entry_timestamp","entry_seq","exit_seq"] if c in a and c in pre]
g=pre.loc[goldmask].merge(a,on=keys,suffixes=("_old","_new"))
print("[GOLD20 JOIN]",len(g))
reg=[]
for p in POLICIES:
    ot=p+"__trigger_old"; nt=p+"__v3_trigger"
    on=p+"__net_return_old"; nn=p+"__v3_net_return"
    trig_match=int((g[ot].fillna(False).astype(bool)==g[nt].fillna(False).astype(bool)).sum())
    neterr=float((pd.to_numeric(g[on],errors="coerce")-pd.to_numeric(g[nn],errors="coerce")).abs().max())
    reg.append((p,trig_match,neterr))
    print("[REGRESSION]",p,"trigger_match",f"{trig_match}/20","net_maxerr",neterr)

# Metrics use ALL 889 trades: ineligible/nontrigger remain original FIXED_4 net, matching original code behavior.
def metrics(df,col):
    z=df.sort_values(["entry_timestamp","symbol"])
    x=pd.to_numeric(z[col],errors="coerce").dropna().astype(float)
    eq=(1+x).cumprod(); dd=eq/eq.cummax()-1
    return dict(trades=len(x),cum_return=float((1+x).prod()-1),log_growth=float(np.log1p(x.clip(lower=-.999999)).sum()),
                mdd=float(dd.min()),mean_trade_return=float(x.mean()),median_trade_return=float(x.median()),win_rate=float((x>0).mean()))

def daily_delta(df,col):
    z=df.copy(); z["_d"]=pd.to_datetime(z.entry_timestamp,utc=True).dt.tz_convert("America/New_York").dt.date
    b=z.groupby("_d").net_return.sum().clip(lower=-.999999)
    c=z.groupby("_d")[col].sum().clip(lower=-.999999)
    ix=b.index.union(c.index)
    return np.log1p(c.reindex(ix,fill_value=0))-np.log1p(b.reindex(ix,fill_value=0))

def boot(df,col,n=10000,seed=42):
    v=daily_delta(df,col).dropna().to_numpy(float); rng=np.random.default_rng(seed)
    bs=np.array([rng.choice(v,size=len(v),replace=True).mean() for _ in range(n)])
    return dict(days=len(v),obs_mean_daily_log_delta=float(v.mean()),ci95_low=float(np.quantile(bs,.025)),
                ci95_high=float(np.quantile(bs,.975)),p_one_sided=float((np.sum(bs<=0)+1)/(n+1)))

base=metrics(a,"net_return")
rows=[]; folds=[]
for p,(lat,_) in POLICIES.items():
    col=p+"__v3_net_return"; m=metrics(a,col); b=boot(a,col)
    pf=0; fc=0
    for fold,part in a.groupby("fold",dropna=False):
        d=float(metrics(part,col)["log_growth"]-metrics(part,"net_return")["log_growth"])
        folds.append(dict(policy=p,fold=str(fold),trades=len(part),paired_log_delta=d))
        pf+=d>0; fc+=1
    rows.append(dict(policy=p,latency_minutes=lat,triggered_exits=int(a[p+"__v3_trigger"].sum()),
                     trigger_rate_eligible=float(a[p+"__v3_trigger"].sum()/eligible.sum()),**m,**b,
                     positive_folds=int(pf),fold_count=int(fc),
                     delta_log_growth_vs_fixed4=float(m["log_growth"]-base["log_growth"]),
                     mdd_delta_vs_fixed4=float(m["mdd"]-base["mdd"])))

# Holm adjustment exactly over the eight candidate p-values.
order=sorted(range(len(rows)),key=lambda i:rows[i]["p_one_sided"])
running=0.0; M=len(rows)
for rank,i in enumerate(order):
    adj=min(1.0,(M-rank)*rows[i]["p_one_sided"]); running=max(running,adj); rows[i]["holm_p"]=running
for x in rows:
    x["research_survivor"]=bool(x["ci95_low"]>0 and x["holm_p"]<=.10 and x["positive_folds"]>=math.ceil(x["fold_count"]*.60)
                                and x["mdd_delta_vs_fixed4"]>=-.02 and eligible.mean()>=.90)

summary=pd.DataFrame(rows).sort_values(["research_survivor","obs_mean_daily_log_delta"],ascending=[False,False])
fold_df=pd.DataFrame(folds)
print("\n[BASELINE]",base)
print("\n[CANDIDATE SUMMARY]")
print(summary.to_string(index=False))

# Freeze outputs separately; canonical audit is NOT overwritten.
stamp=datetime.now(timezone.utc).isoformat()
summary.to_csv(OUT/"candidate_summary_v3_0.csv",index=False)
fold_df.to_csv(OUT/"fold_summary_v3_0.csv",index=False)
cols=keys+["net_return","weight","revalidation_data_ready"]+[c for p in POLICIES for c in (p+"__v3_trigger",p+"__v3_net_return",p+"__v3_exit_price")]
a[cols].to_parquet(OUT/"trade_audit_v3_0.parquet",index=False)
report={"schema":"kalman-open-revalidation-frozen-original-contract-v3.0","generated_at_utc":stamp,
"source_commit":"f56d51c795989c33961ee98cfd11fffefae235b5","research_only":True,"production_changed":False,
"live_trading_changed":False,"neon_write":False,"rows":len(a),"eligible_rows":int(eligible.sum()),
"coverage":float(eligible.mean()),"gold20_regression":[{"policy":p,"trigger_match":t,"net_maxerr":e} for p,t,e in reg],
"baseline":base,"candidates":rows,"survivors":[x["policy"] for x in rows if x["research_survivor"]],
"recommendation":"PROSPECTIVE_SHADOW_ONLY" if any(x["research_survivor"] for x in rows) else "NO_SURVIVOR_KEEP_BLOCKED"}
(OUT/"decision_v3_0.json").write_text(json.dumps(report,indent=2,default=str)+"\n")
print("\n[DECISION]",report["recommendation"],"survivors=",report["survivors"])
print("[OUTPUT]",OUT)
print("READ/RESEARCH ONLY relative to production. Canonical audit not overwritten.")
